<a href="https://colab.research.google.com/github/Dacon-TeamPK/dacon-physics-vision-ai/blob/model/resnet/ResNet50_0.39.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MultiResNet50



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. import+CFG

In [ ]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from tqdm.auto import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 224,
    'STAGE1_EPOCHS': 3,      # classifier만 학습
    'STAGE2_EPOCHS': 10,     # 전체 fine-tuning
    'BACKBONE_LR': 1e-5,
    'HEAD_LR': 1e-4,
    'BATCH_SIZE': 24,
    'SEED': 42,
    'DROPOUT': 0.3
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
os.chdir('/content/drive/MyDrive/open_dacon')

print(os.getcwd())

## 2. 데이터 로드

In [ ]:
train_df = pd.read_csv('./train.csv')
val_df = pd.read_csv('./dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

## 3.custom dataset class, data loader

In [ ]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        folder_path = os.path.join(self.root_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views

        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = MultiViewDataset(train_df, './train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, './dev', test_transform, is_test=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_df = pd.read_csv('./sample_submission.csv')
test_dataset = MultiViewDataset(test_df, './test', test_transform, is_test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

## 4. 모델 정의

In [ ]:
class MultiViewResNet50(nn.Module):
    def __init__(self, num_classes=1, dropout=0.3):
        super(MultiViewResNet50, self).__init__()

        self.backbone = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        feat_dim = 2048

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes)
        )

    def forward(self, views):
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        diff = torch.abs(f1 - f2)
        prod = f1 * f2

        combined = torch.cat([f1, f2, diff, prod], dim=1)
        return self.classifier(combined)

In [ ]:
def freeze_backbone(model):
    for param in model.feature_extractor.parameters():
        param.requires_grad = False

def unfreeze_backbone(model):
    for param in model.feature_extractor.parameters():
        param.requires_grad = True

In [ ]:
def get_optimizer_stage1(model):
    # classifier만 학습
    optimizer = optim.AdamW(
        model.classifier.parameters(),
        lr=CFG['HEAD_LR'],
        weight_decay=1e-4
    )
    return optimizer

def get_optimizer_stage2(model):
    # backbone과 classifier lr 분리
    optimizer = optim.AdamW([
        {'params': model.feature_extractor.parameters(), 'lr': CFG['BACKBONE_LR']},
        {'params': model.classifier.parameters(), 'lr': CFG['HEAD_LR']}
    ], weight_decay=1e-4)
    return optimizer

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0.0

    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(loader)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return val_loss / len(loader), logloss_score, acc_score

In [ ]:
model = MultiViewResNet50(dropout=CFG['DROPOUT']).to(device)
criterion = nn.BCEWithLogitsLoss()
freeze_backbone(model)
optimizer = get_optimizer_stage1(model)

best_logloss = float('inf')
best_epoch = 0
global_epoch = 0

print("===== Stage 1: classifier only training =====")

for epoch in range(1, CFG['STAGE1_EPOCHS'] + 1):
    global_epoch += 1

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_logloss, val_acc = validate(model, val_loader, criterion, device)

    print(f"\n[Stage 1] Epoch [{epoch}/{CFG['STAGE1_EPOCHS']}]")
    print(f"  - Train Loss   : {train_loss:.4f}")
    print(f"  - Val Loss     : {val_loss:.4f}")
    print(f"  - Val LogLoss  : {val_logloss:.6f}")
    print(f"  - Val Acc      : {val_acc:.4f}")

    if val_logloss < best_logloss:
        best_logloss = val_logloss
        best_epoch = global_epoch
        torch.save(model.state_dict(), "best_model_resnet50_single.pth")
        print("  -> Best model saved!")

unfreeze_backbone(model)
optimizer = get_optimizer_stage2(model)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

print("\n===== Stage 2: full fine-tuning =====")

for epoch in range(1, CFG['STAGE2_EPOCHS'] + 1):
    global_epoch += 1

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_logloss, val_acc = validate(model, val_loader, criterion, device)

    print(f"\n[Stage 2] Epoch [{epoch}/{CFG['STAGE2_EPOCHS']}]")
    print(f"  - Train Loss   : {train_loss:.4f}")
    print(f"  - Val Loss     : {val_loss:.4f}")
    print(f"  - Val LogLoss  : {val_logloss:.6f}")
    print(f"  - Val Acc      : {val_acc:.4f}")
    print(f"  - Backbone LR  : {optimizer.param_groups[0]['lr']:.8f}")
    print(f"  - Head LR      : {optimizer.param_groups[1]['lr']:.8f}")

    scheduler.step(val_logloss)

    if val_logloss < best_logloss:
        best_logloss = val_logloss
        best_epoch = global_epoch
        torch.save(model.state_dict(), "best_model_resnet50_single.pth")
        print("  -> Best model saved!")

In [ ]:
print(f"\nBest Epoch: {best_epoch}")
print(f"Best Val LogLoss: {best_logloss:.6f}")

## 5. 추론 및 제출 파일 생성

In [ ]:
model = MultiViewResNet50(dropout=CFG['DROPOUT']).to(device)
model.load_state_dict(torch.load("best_model_resnet50_single.pth", map_location=device))
model.eval()

all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference"):
        views = [v.to(device) for v in views]

        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)

submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

submission.to_csv('submission_resnet50_single_pipeline.csv', encoding='UTF-8-sig', index=False)
print("submission_resnet50_single_pipeline.csv 저장 완료.")

In [ ]:
print("Best Val LogLoss:", best_logloss)